# Spark on Kubernetes: Cloud-Native Pod-Based Architecture

**Module**: 12_advanced_topics  
**Notebook**: 02_spark_on_kubernetes.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Apache Spark runs natively on Kubernetes: Driver pods, Executor pods, Spark Operator, and cloud-native scaling.
- **Why does this topic exist?**: Kubernetes is rapidly becoming the modern standard for cloud-native big data infrastructure.
- **What problem does it solve?**: Managing ephemeral pod lifecycles and Docker container images.
- **Where is it used in real-world Data Engineering?**: Running Spark on EKS, GKE, and AKS.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Containers, Kubernetes basics, Spark architecture |
| **Required depth** | Deep working knowledge: can explain the driver/executor pod lifecycle and resource configuration |
| **Data Engineer** | Submit and operate Spark jobs on Kubernetes. |
| **Senior Data Engineer** | Configure pod templates, service accounts, and resource requests. |
| **Spark specialist** | Reason about dynamic allocation, shuffle storage, and pod scheduling constraints. |
| **Next topic** | `12_advanced_topics/03_cluster_vs_client_mode.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spark on Kubernetes runs the driver and executors as pods. The Spark submission client talks to the
Kubernetes API server, which schedules pods according to resource requests, node selectors, and quotas. This gives
strong isolation and elastic scaling, but it also means Spark must handle pod scheduling delays, ephemeral
storage, and the loss of executor-local shuffle data.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Driver pod** | Runs the Spark driver | Created by the submission client or a controller |
| **Executor pods** | Run executors | Created on demand and removed when idle |
| **Service account** | API identity | Needs permissions to create/watch pods |
| **Pod template** | Pod customization | Node selectors, volumes, tolerations, sidecars |
| **Resource request** | CPU/memory request | Used by the scheduler to place pods |
| **Remote shuffle** | Shuffle storage | Required when pods are ephemeral and shuffle data must survive |

### How it works, step by step

1. The client builds the driver pod specification.
2. It submits the pod to the Kubernetes API server.
3. The driver starts and requests executor pods.
4. The scheduler places executor pods on nodes with available resources.
5. Executors register with the driver and process tasks.
6. Idle executors are deleted; shuffle data may need a remote shuffle service to survive.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Spark on Kubernetes Architecture
```
Kubernetes Control Plane (kube-apiserver):
  1. spark-submit --master k8s://https://k8s-apiserver:443
  2. K8s launches Driver Pod.
  3. Driver Pod requests Executor Pods directly from kube-apiserver.
  4. K8s schedules Executor Pods on available K8s nodes.
  5. Executor Pods connect back to Driver Pod via K8s headless service.
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("K8s_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("K8s submission equivalent command:")
print("""
spark-submit \
  --master k8s://https://kubernetes.default.svc \
  --deploy-mode cluster \
  --name spark-k8s-job \
  --conf spark.kubernetes.container.image=spark-local:4.2.1 \
  --conf spark.kubernetes.driver.pod.name=spark-driver-pod \
  --conf spark.executor.instances=4 \
  local:///opt/spark-apps/sales_report.py
""")

---

## Experiment: Comparing K8s vs Standalone

- **1. What are we testing?**: Contrast K8s pod lifecycle with Docker Compose standalone.
- **2. What do we expect?**: K8s creates ephemeral pods on demand; Standalone uses persistent worker daemons.

In [ ]:
print('K8s provides fine-grained container isolation and auto-scaling via Cluster Autoscaler.')

**3. What happened?**: Explained K8s pod dynamics.  
**4. Why did it happen?**: K8s provisions dedicated containers per executor.  
**5. Practical Engineering Lesson**: K8s eliminates idle worker costs through cloud autoscaling.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Elastic batch platform

- **Industry**: SaaS
- **Scenario**: A company runs Spark jobs on a shared Kubernetes cluster with autoscaling.
- **Problem**: Jobs need to scale with data volume without permanently reserving nodes.
- **Why this concept is relevant**: Executor pods are created on demand and removed when idle, matching cost to workload.
- **Implementation idea**: Configure dynamic allocation, resource requests, and pod templates for the shared cluster.
- **Expected result**: Elastic capacity with lower idle cost.
- **Alternative**: Run a dedicated Spark cluster.
- **Trade-offs**: A dedicated cluster is simpler to operate but wastes capacity outside peak periods.

### Use Case 2: Multi-tenant isolation

- **Industry**: Financial services
- **Scenario**: Multiple teams share a Kubernetes cluster and need resource isolation.
- **Problem**: A runaway job can starve other teams or consume all node memory.
- **Why this concept is relevant**: Namespaces, quotas, and resource requests/limits provide isolation and predictable scheduling.
- **Implementation idea**: Use namespaces per team, set quotas, and define pod templates with requests and limits.
- **Expected result**: Fair sharing with bounded blast radius.
- **Alternative**: Separate clusters per team.
- **Trade-offs**: Separate clusters isolate strongly but multiply operational overhead and cost.

---

## Business Value

- Elastic infrastructure aligned with workload demand.
- Strong isolation between teams.
- Portable deployment across cloud providers.
- Clear resource ownership through Kubernetes quotas.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Set resource requests and limits; the scheduler uses requests.
- Use a service account with least privilege.
- Plan for remote shuffle storage when pods are ephemeral.
- Use pod templates for node selectors, tolerations, and sidecars.
- Monitor pod scheduling delays and evictions.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Inspect the pod lifecycle

**Steps**

1. Submit a small Spark job to Kubernetes.
2. Watch the driver pod and executor pods with `kubectl get pods`.
3. Inspect events for scheduling, image pull, and termination.

**What to observe**: The driver creates executor pods on demand and deletes them when idle.

**Healthy result**: Pods start within the expected time and register with the driver.

**Problematic result**: Executor pods remain Pending due to insufficient resources or quota.

**Engineering takeaway**: Spark scheduling depends on Kubernetes scheduling succeeding first.

---

## Trade-offs and Alternatives

### Advantages

- Elastic scaling
- Isolation
- Cloud portability

### Limitations

- Operational complexity
- Ephemeral shuffle storage
- Scheduling delays

### When to use it

- Cloud-native platforms
- Multi-tenant clusters
- Elastic batch

### When not to use it

- Assuming executor-local shuffle survives
- Ignoring resource requests
- Overly broad service accounts

### Alternatives inside Spark

- Driver/executor pods
- Dynamic allocation
- Pod templates

### Alternatives outside Spark

- YARN
- Standalone mode
- Managed Spark services

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): K8s Master URL Syntax
**Problem Statement**: What is the syntax for Spark K8s master?  
**Input / Context**: URL

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Spark Operator Concept
**Problem Statement**: Explain Spark on K8s Operator.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Driver Headless Service
**Problem Statement**: Why K8s requires a headless service for Driver.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Executors stuck in Pending

**Scenario**: A Spark job's driver is running, but executor pods remain Pending.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check pod events for insufficient CPU/memory or quota limits.
2. Compare requests with node allocatable resources.
3. Inspect node selectors, taints, and tolerations.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (K8s Master URL Syntax):
```python
print('k8s://https://<k8s-apiserver-host>:<port>')
```

#### Solution 2 (Spark Operator Concept):
```python
print('CRD (Custom Resource Definition) allowing declarative SparkApplication YAML manifests.')
```

#### Solution 3 (Driver Headless Service):
```python
print('Enables executor pods to route Netty RPC traffic back to the dynamic driver pod IP.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Cloud-Native**: Pods as executors.
- **Autoscaling**: Integrates with K8s cluster autoscaler.

### Essential Technical Terms
- `Driver Pod`: Pod hosting driver process.
- `Spark Operator`: Kubernetes CRD controller for Spark.

### Common Engineering Mistakes
- **Missing ServiceAccount RBAC permissions**: Driver pod fails to launch executor pods if K8s RBAC is missing.

### Interview Questions on This Topic
- **Q: How does the Driver communicate with executors on Kubernetes?**
  - *A*: Through a Kubernetes headless service that routes Netty RPC packets directly to executor pod IPs.


### More Interview Questions

- **Q: How does Spark on Kubernetes create executors?**
  - *A*: The driver requests executor pods through the Kubernetes API; the scheduler places them on nodes with sufficient resources, and they register with the driver.
- **Q: Why is remote shuffle storage often needed on Kubernetes?**
  - *A*: Executor pods are ephemeral; when a pod is deleted, its local shuffle data is lost, so a remote shuffle service can preserve it for recovery and dynamic allocation.

### What to Study Next
Open 03_cluster_vs_client_mode.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')